NOT TESTED YET

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader, random_split

In [ ]:
Zmumu = pd.read_parquet("/data/delphes_Zmumu.parquet")
features = Zmumu
targets = Zmumu["event_type"]

In [ ]:
dataset = torch.utils.data.TensorDataset(
    torch.tensor(features.values, dtype=torch.float32),
    torch.tensor(targets.cat.codes.values, dtype=torch.int64),
)

In [ ]:
subset_train, subset_valid, subset_test = torch.utils.data.random_split(dataset, [0.8, 0.1, 0.1])

In [ ]:
class ScaleInputs(nn.Module):
    def __init__(self, subset_train):
        super().__init__()

        # get a single features tensor for the whole training subset
        ((features_tensor, targets_tensor),) = DataLoader(subset_train, batch_size=len(targets))

        # get 16 means and 16 standard deviations from the training dataset
        self.register_buffer("means", features_tensor.mean(axis=0))
        self.register_buffer("stds", features_tensor.std(axis=0))

    def forward(self, x):
        return (x - self.means) / self.stds

In [ ]:
scale_inputs = ScaleInputs(subset_train)

In [ ]:
model_without_softmax = nn.Sequential(
    scale_inputs,
    nn.Linear(25, 50),   # 16 input features → a hidden layer with 32 neurons
    nn.ReLU(),           # ReLU avoids the vanishing gradients problem
    nn.Linear(50, 50),   # first hidden layer → second hidden layer
    nn.ReLU(),
    nn.Linear(50, 50),   # second hidden layer → third hidden layer
    nn.ReLU(),
    nn.Linear(50, 2),    # third hidden layer → 5 output category probabilities
    # no softmax because CrossEntropyLoss automatically applies it
)

In [ ]:
NUM_EPOCHS = 10
BATCH_SIZE = 20000

train_loader = DataLoader(subset_train, batch_size=BATCH_SIZE)
valid_loader = DataLoader(subset_valid, batch_size=len(subset_valid))

loss_function = nn.CrossEntropyLoss()

optimizer = optim.Adam(model_without_softmax.parameters(), lr=0.03)

valid_loss_vs_epoch = []
train_loss_vs_epoch = []

for epoch in range(NUM_EPOCHS):
    ((features_tensor, targets_tensor),) = valid_loader

    predictions_tensor = model_without_softmax(features_tensor)
    loss = loss_function(predictions_tensor, targets_tensor)
    valid_loss = loss.item() * len(targets_tensor) * (0.8 / 0.1)  

    valid_loss_vs_epoch.append(valid_loss)

    train_loss = 0
    for features_tensor, targets_tensor in train_loader:
        optimizer.zero_grad()
    
        predictions_tensor = model_without_softmax(features_tensor)
        loss = loss_function(predictions_tensor, targets_tensor)
        train_loss += loss.item() * len(targets_tensor)           
    
        loss.backward()
        optimizer.step()

    train_loss_vs_epoch.append(train_loss)

    print(f"{epoch = } {train_loss = } {valid_loss = }")

In [ ]:
fig, ax = plt.subplots()

ax.plot(range(len(train_loss_vs_epoch)), train_loss_vs_epoch, marker=".")
ax.plot(range(len(valid_loss_vs_epoch)), valid_loss_vs_epoch, marker=".")

ax.grid(True, "both", "y", linestyle=":")

ax.set_xticks(range(NUM_EPOCHS))
ax.set_ylim(0, 1.2*max(max(train_loss_vs_epoch), max(valid_loss_vs_epoch)))
ax.set_xlabel("epoch number")
ax.set_ylabel("loss")
ax.legend(["training sample", "validation sample"])

plt.show()

In [ ]:
model_with_softmax = nn.Sequential(
    model_without_softmax,
    nn.Softmax(dim=1),
)

In [ ]:
test_loader = DataLoader(subset_test, batch_size=len(subset_test))

((features_tensor, targets_tensor),) = test_loader
predictions_tensor = model_with_softmax(features_tensor)

confusion_matrix = np.array(
    [
        [
            (predictions_tensor[targets_tensor == true_class].argmax(axis=1) == prediction_class).sum().item()
            for prediction_class in range(2)
        ]
        for true_class in range(2)
    ]
)
confusion_matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

image = ax.imshow(confusion_matrix, vmin=0)
fig.colorbar(image, ax=ax, label="number of test samples", shrink=0.8)

ax.set_xticks(range(5), targets.cat.categories)
ax.set_yticks(range(5), targets.cat.categories)

ax.set_xlabel("predicted process category")
ax.set_ylabel("true process category")

None

In [ ]:
def matrix_for_cutoff_decision_at(threshold):
    is_signal = (targets_tensor >= 1)

    true_positive  = (predictions_tensor[is_signal][:, 2:5].sum(axis=1) > threshold).sum().item()
    false_positive = (predictions_tensor[~is_signal][:, 2:5].sum(axis=1) > threshold).sum().item()

    false_negative = (predictions_tensor[is_signal][:, 2:5].sum(axis=1) <= threshold).sum().item()
    true_negative  = (predictions_tensor[~is_signal][:, 2:5].sum(axis=1) <= threshold).sum().item()

    return np.array([
        [true_positive, false_positive],
        [false_negative, true_negative],
    ])

In [ ]:
true_positive_rates = []
false_positive_rates = []

for threshold in np.linspace(0, 1, 1000):
    ((true_positive, false_positive),
     (false_negative, true_negative)) = matrix_for_cutoff_decision_at(threshold)

    true_positive_rates.append(true_positive / (true_positive + false_negative))
    false_positive_rates.append(false_positive / (true_negative + false_positive))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.plot(false_positive_rates, true_positive_rates, label="this model")

ax.grid(True, linestyle=":")

ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")

ax.legend(loc="lower right")

plt.show()